# Digital Twin ML Training -- Anomaly Detection, Fault Classification, RUL

Trains three models on simulated MALE UAV piston engine telemetry:
1. **Anomaly detector** (Isolation Forest, trained only on healthy data)
2. **Fault classifier** (XGBoost multi-class: healthy / misfire / cooling_degradation / lubrication_degradation / sensor_drift_cht)
3. **RUL regressor** (XGBoost regression, predicts seconds remaining until failure)

Plus SHAP explainability so predictions come with a "why", not just a label.

**Upload your dataset CSV as a Kaggle Dataset input before running.**

In [6]:
!pip install xgboost shap scikit-learn joblib -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
)
import xgboost as xgb
import shap
import joblib
import os

warnings.filterwarnings("ignore")

SENSOR_COLS = ["rpm", "cht", "egt", "oil_pressure", "oil_temp", "fuel_flow", "vibration", "battery_voltage"]
ROLLING_WINDOW = 6  # ~1 minute of history at 10s sampling

## 1. Load and clean data

Point `DATA_PATH` at your uploaded Kaggle dataset.

In [8]:
# EDIT THIS to your actual Kaggle input path, e.g.:
# DATA_PATH = "/kaggle/input/uav-training-dataset/uav_training_data.csv"
DATA_PATH = r"C:\Users\Ansh Mishra\Desktop\SIH\uav_training_data.csv"

def load_and_clean(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Defensive fix: a stray shell command has occasionally been seen
    # prepended to the first column's header in some exports. If the first
    # column doesn't look like 'flight_id', fix it.
    if df.columns[0] != "flight_id":
        df = df.rename(columns={df.columns[0]: "flight_id"})
    df = df.sort_values(["flight_id", "t_s"]).reset_index(drop=True)
    return df

df = load_and_clean(DATA_PATH)
print(f"Loaded {len(df):,} rows, {df.flight_id.nunique()} flights")
df.head()

Loaded 368,028 rows, 150 flights


,flight_id,t_s,phase,throttle_pct,altitude_m,ambient_c,rpm,map_inHg,cht,egt,oil_pressure,oil_temp,fuel_flow,vibration,battery_voltage,fault_label,remaining_useful_life
0,0,0.0,taxi,15.3,0.0,40.0,2513.196,14.313,46.132,407.841,3.697,41.682,4.686,0.651,13.702,healthy,NaN
1,0,10.0,taxi,15.9,0.0,40.0,2544.307,14.771,52.631,462.528,3.651,43.495,4.980,0.665,13.823,healthy,NaN
2,0,20.0,taxi,14.8,0.0,40.0,2485.028,14.304,58.618,464.557,3.646,44.949,4.758,0.595,13.779,healthy,NaN
3,0,30.0,taxi,17.1,0.0,40.0,2626.515,14.742,64.691,472.279,3.648,46.214,5.201,0.571,13.831,healthy,NaN
4,0,40.0,taxi,15.7,0.0,40.0,2523.532,14.702,69.116,470.986,3.608,47.952,4.928,0.619,13.806,healthy,NaN


## 2. Physics-informed expected values

Mirrors the simulator's recalibrated + turbocharged physics model (steady-state only). This gives us **residual features**: actual minus expected, which is far more informative than raw sensor values alone.

In [9]:
IDLE_RPM, MAX_RPM, CRUISE_RPM = 1400.0, 5800.0, 5000.0
MAP_IDLE, MAP_MAX = 12.0, 29.9
CHT_IDLE, CHT_CRUISE, CHT_REDLINE = 90.0, 125.0, 150.0
EGT_IDLE, EGT_CRUISE, EGT_REDLINE = 420.0, 800.0, 900.0
OIL_P_MIN, OIL_P_MAX = 0.8, 5.0
OIL_T_IDLE, OIL_T_CRUISE = 60.0, 100.0
FUEL_FLOW_MAX = 25.0
VIB_BASELINE = 1.0
BATTERY_NOMINAL = 13.8
TURBO_CRITICAL_ALT_M = 5000.0
CRUISE_LOAD_REF = 0.7


def expected_engine_values(throttle_pct, altitude_m, ambient_c):
    thr = throttle_pct / 100.0
    temp_k = ambient_c + 273.15
    pressure_pa = 101325 * (1 - 2.25577e-5 * altitude_m) ** 5.25588

    above_critical = altitude_m > TURBO_CRITICAL_ALT_M
    excess_alt = np.clip(altitude_m - TURBO_CRITICAL_ALT_M, 0, None)
    falloff = np.clip((1 - 2.25577e-5 * excess_alt) ** 5.25588, 0.3, None)
    turbo_map_ceiling = np.where(above_critical, MAP_MAX * falloff, MAP_MAX)
    turbo_density_ratio = np.where(above_critical, falloff, 1.0)

    map_expected = MAP_IDLE + thr * (turbo_map_ceiling - MAP_IDLE)

    rpm_expected = np.where(
        thr <= 0.5,
        IDLE_RPM + (thr / 0.5) * (CRUISE_RPM - IDLE_RPM),
        CRUISE_RPM + ((thr - 0.5) / 0.5) * (MAX_RPM - CRUISE_RPM),
    )

    load_factor = np.clip((map_expected / MAP_MAX) * (rpm_expected / MAX_RPM), 0, None)

    frac_low = load_factor / CRUISE_LOAD_REF
    frac_high = np.clip((load_factor - CRUISE_LOAD_REF) / (1.0 - CRUISE_LOAD_REF), None, 1.2)
    egt_expected = np.where(load_factor <= CRUISE_LOAD_REF,
                             EGT_IDLE + frac_low * (EGT_CRUISE - EGT_IDLE),
                             EGT_CRUISE + frac_high * (EGT_REDLINE - EGT_CRUISE))
    cht_expected = np.where(load_factor <= CRUISE_LOAD_REF,
                             CHT_IDLE + frac_low * (CHT_CRUISE - CHT_IDLE),
                             CHT_CRUISE + frac_high * (CHT_REDLINE - CHT_CRUISE))

    ambient_offset = ambient_c - 15.0
    ram_air_cooling = thr * 15.0
    cht_expected = cht_expected + ambient_offset - ram_air_cooling

    oil_temp_expected = OIL_T_IDLE + (OIL_T_CRUISE - OIL_T_IDLE) * load_factor * 1.2
    oil_temp_expected = oil_temp_expected + ambient_offset * 0.5 - ram_air_cooling * 0.5

    rpm_factor = rpm_expected / MAX_RPM
    oil_pressure_expected = OIL_P_MIN + (OIL_P_MAX - OIL_P_MIN) * rpm_factor
    oil_pressure_expected = np.clip(oil_pressure_expected, None, 7.0)

    vibration_expected = VIB_BASELINE * (0.5 + 0.5 * load_factor)
    fuel_flow_expected = FUEL_FLOW_MAX * load_factor * turbo_density_ratio
    battery_expected = np.where(rpm_expected < 2000, BATTERY_NOMINAL - 0.6, BATTERY_NOMINAL)

    return pd.DataFrame({
        "rpm_expected": rpm_expected, "cht_expected": cht_expected, "egt_expected": egt_expected,
        "oil_pressure_expected": oil_pressure_expected, "oil_temp_expected": oil_temp_expected,
        "fuel_flow_expected": fuel_flow_expected, "vibration_expected": vibration_expected,
        "battery_voltage_expected": battery_expected,
    })

## 3. Feature engineering

- **Residuals**: actual minus physics-expected value per sensor
- **Rolling stats**: mean/std over the last ~1 minute, computed per-flight so history never leaks across flights
- **Rate of change**: how fast each sensor is moving
- **Time in anomaly state**: how long a deviation has persisted (computed WITHOUT looking at ground-truth labels, so it works identically on real live data at inference time) -- this matters a lot for RUL, since remaining life depends on the *rate* of degradation, not just its current magnitude

In [10]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    expected = expected_engine_values(df["throttle_pct"].values, df["altitude_m"].values, df["ambient_c"].values)
    for col in SENSOR_COLS:
        df[f"{col}_residual"] = df[col].values - expected[f"{col}_expected"].values

    grouped = df.groupby("flight_id", sort=False)
    for col in SENSOR_COLS:
        df[f"{col}_roll_mean"] = grouped[col].transform(lambda s: s.rolling(ROLLING_WINDOW, min_periods=1).mean())
        df[f"{col}_roll_std"] = grouped[col].transform(lambda s: s.rolling(ROLLING_WINDOW, min_periods=1).std().fillna(0))
        df[f"{col}_rate"] = grouped[col].transform(lambda s: s.diff().fillna(0))

    is_anomalous = (
        (df["oil_pressure_residual"].abs() > 0.5) |
        (df["cht_residual"].abs() > 15) |
        (df["egt_residual"].abs() > 50) |
        (df["vibration_residual"].abs() > 0.5)
    )
    df["_is_anomalous"] = is_anomalous
    df["time_in_anomaly_state"] = df.groupby("flight_id", group_keys=False)["_is_anomalous"].apply(
        lambda s: s.groupby((~s).cumsum()).cumcount().where(s, 0)
    )
    df = df.drop(columns=["_is_anomalous"])
    return df


def get_feature_columns():
    cols = []
    for col in SENSOR_COLS:
        cols += [f"{col}_residual", f"{col}_roll_mean", f"{col}_roll_std", f"{col}_rate"]
    cols += ["throttle_pct", "altitude_m", "ambient_c", "time_in_anomaly_state"]
    return cols


df = add_features(df)
feature_cols = get_feature_columns()
print(f"{len(feature_cols)} features engineered")

36 features engineered


## 4. Split by flight (not by row!)

Rows within a flight are highly correlated second-to-second. Randomly shuffling rows into train/test would let the model "cheat" by seeing near-identical rows from the same flight in both sets. We split by whole flights instead -- the model never sees any row from a test flight during training.

In [11]:
def split_by_flight(df, test_frac=0.2, seed=42):
    flight_ids = df["flight_id"].unique()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(flight_ids)
    n_test = max(1, int(len(shuffled) * test_frac))
    test_ids = set(shuffled[:n_test])
    train_ids = set(shuffled[n_test:])
    return df[df.flight_id.isin(train_ids)].copy(), df[df.flight_id.isin(test_ids)].copy()

train_df, test_df = split_by_flight(df, test_frac=0.2, seed=42)
print(f"Train: {train_df.flight_id.nunique()} flights, {len(train_df):,} rows")
print(f"Test:  {test_df.flight_id.nunique()} flights, {len(test_df):,} rows")

Train: 120 flights, 288,277 rows
Test:  30 flights, 79,751 rows


## 5. Model 1 -- Anomaly Detector

Trained **only on healthy rows**. It learns what "normal" looks like and flags anything that deviates -- including fault types it's never explicitly seen, which matters in the real world where you can't anticipate every failure mode in advance.

In [12]:
# --- PRO LEVEL FIX: Model 1 -- Anomaly Detector (Fully Self-Contained) ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

# 1. Split data by flight (If not already done)
flights = df['flight_id'].unique()
train_flights, test_flights = train_test_split(flights, test_size=0.2, random_state=42)
train_df = df[df['flight_id'].isin(train_flights)]
test_df = df[df['flight_id'].isin(test_flights)]

# 2. Generate Physics Residuals using YOUR physics engine
# (Make sure the cell with expected_engine_values has been run before this)
expected_train = expected_engine_values(train_df['throttle_pct'], train_df['altitude_m'], train_df['ambient_c'])
train_df['egt_resid'] = train_df['egt'] - expected_train['egt_expected']
train_df['cht_resid'] = train_df['cht'] - expected_train['cht_expected']
train_df['oil_p_resid'] = train_df['oil_pressure'] - expected_train['oil_pressure_expected']
train_df['vib_resid'] = train_df['vibration'] - expected_train['vibration_expected']
train_df['rpm_resid'] = train_df['rpm'] - expected_train['rpm_expected']

expected_test = expected_engine_values(test_df['throttle_pct'], test_df['altitude_m'], test_df['ambient_c'])
test_df['egt_resid'] = test_df['egt'] - expected_test['egt_expected']
test_df['cht_resid'] = test_df['cht'] - expected_test['cht_expected']
test_df['oil_p_resid'] = test_df['oil_pressure'] - expected_test['oil_pressure_expected']
test_df['vib_resid'] = test_df['vibration'] - expected_test['vibration_expected']
test_df['rpm_resid'] = test_df['rpm'] - expected_test['rpm_expected']

# 3. THE NEW FEATURE_COLS (Only residuals!)
feature_cols = ['egt_resid', 'cht_resid', 'oil_p_resid', 'vib_resid', 'rpm_resid']

# 4. Scale the features
scaler = StandardScaler()
healthy = train_df[train_df.fault_label == "healthy"]
X_healthy = scaler.fit_transform(healthy[feature_cols])
X_test = scaler.transform(test_df[feature_cols])

# 5. Train Isolation Forest
anomaly_model = IsolationForest(n_estimators=500, contamination=0.01, random_state=42, n_jobs=-1)
anomaly_model.fit(X_healthy)

# 6. Calculate Adaptive Threshold (80th percentile)
healthy_scores = -anomaly_model.score_samples(X_healthy)
threshold = np.percentile(healthy_scores, 80)

# 7. Apply to Test Set
test_scores = -anomaly_model.score_samples(X_test)
pred_anomaly = (test_scores > threshold).astype(int)

# 8. Evaluate
y_true = (test_df["fault_label"] != "healthy").astype(int).values
auc = roc_auc_score(y_true, test_scores)
f1 = f1_score(y_true, pred_anomaly)

print(f"ROC-AUC: {auc:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Flagged {pred_anomaly.sum()} / {len(pred_anomaly)} rows as anomalous (true fault rate: {y_true.mean():.1%})")

ROC-AUC: 0.865
F1 Score: 0.717
Flagged 29542 / 73900 rows as anomalous (true fault rate: 27.8%)


## 6. Model 2 -- Fault Classifier

Multi-class XGBoost: given the engineered features, which fault (if any) is happening?

**Important nuance**: once an engine has *completely* failed (RPM ~ 0), all degradation faults look identical (oil pressure, fuel flow, etc. all crash to 0 regardless of root cause) -- a single snapshot genuinely can't recover which fault caused a total failure after the fact. This is also not very actionable anyway, since the mission has already ended by that point. We evaluate both the full test set AND the pre-failure-only window, since the pre-failure window is what a real predictive-maintenance system actually needs to get right.

In [ ]:
labels = sorted(train_df["fault_label"].unique())
label_to_idx = {l: i for i, l in enumerate(labels)}
y_train = train_df["fault_label"].map(label_to_idx).values

classifier_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=len(labels),
    random_state=42, n_jobs=-1, eval_metric="mlogloss",
)
classifier_model.fit(train_df[feature_cols], y_train)


def report(subset_df, title):
    if len(subset_df) == 0:
        return
    y_true = subset_df["fault_label"].map(label_to_idx).values
    y_pred = classifier_model.predict(subset_df[feature_cols])
    print(f"--- {title} ({len(subset_df):,} rows) ---")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

report(test_df, "All rows")
report(test_df[test_df["rpm"] > 100], "Pre-failure only (the actionable window)")

y_true_all = test_df["fault_label"].map(label_to_idx).values
y_pred_all = classifier_model.predict(test_df[feature_cols])
cm = confusion_matrix(y_true_all, y_pred_all)
print("Confusion matrix (rows=true, cols=predicted):")
display(pd.DataFrame(cm, index=labels, columns=labels))

NameError: name 'clf' is not defined

In [14]:
# Feature importance -- sanity check that the model is using physically sensible signals
importances = sorted(zip(feature_cols, classifier_model.feature_importances_), key=lambda x: -x[1])
print("Top 10 most important features:")
for name, imp in importances[:10]:
    print(f"  {name:30s} {imp:.4f}")

Top 10 most important features:
  cht_resid                      0.4668
  oil_p_resid                    0.2922
  egt_resid                      0.1118
  vib_resid                      0.0771
  rpm_resid                      0.0520


## 7. Model 3 -- RUL Regressor

Predicts seconds remaining until failure, trained only on rows where a real countdown exists (degradation faults: lubrication and cooling).

In [15]:
rul_train = train_df[train_df["remaining_useful_life"].notna()]
rul_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", random_state=42, n_jobs=-1,
)
rul_model.fit(rul_train[feature_cols], rul_train["remaining_useful_life"])

rul_test = test_df[test_df["remaining_useful_life"].notna()]
y_true = rul_test["remaining_useful_life"].values
y_pred = rul_model.predict(rul_test[feature_cols])
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
print(f"MAE: {mae:.1f}s ({mae/60:.1f} min)   RMSE: {rmse:.1f}s   R2: {r2:.3f}")

MAE: 1876.2s (31.3 min)   RMSE: 2931.0s   R2: 0.435


## 8. SHAP Explainability

Turns a prediction into a human-readable "why" -- this is what your dashboard should show operators, not a bare label.

In [16]:
def explain_prediction(row_features, top_n=3):
    explainer = shap.TreeExplainer(classifier_model)
    shap_values = explainer.shap_values(row_features)

    pred_idx = int(np.argmax(classifier_model.predict_proba(row_features)[0]))
    pred_label = labels[pred_idx]

    if isinstance(shap_values, list):
        values_for_class = shap_values[pred_idx][0]
    else:
        values_for_class = shap_values[0, :, pred_idx] if shap_values.ndim == 3 else shap_values[0]

    feature_names = row_features.columns.tolist()
    contrib = sorted(zip(feature_names, values_for_class), key=lambda x: -abs(x[1]))[:top_n]
    total_abs = sum(abs(v) for _, v in contrib) or 1e-9
    parts = [f"{abs(v)/total_abs:.0%} due to {name}" for name, v in contrib]
    return f"{pred_label} predicted: " + ", ".join(parts)

faulted_rows = test_df[test_df.fault_label != "healthy"]
sample_row = faulted_rows.iloc[[len(faulted_rows) // 2]]
print("Ground truth:", sample_row["fault_label"].values[0])
print("Explanation: ", explain_prediction(sample_row[feature_cols]))

Ground truth: sensor_drift_cht
Explanation:  sensor_drift_cht predicted: 93% due to cht_resid, 4% due to vib_resid, 3% due to egt_resid


## 9. Save trained models

Download these from Kaggle's Output panel -- you'll load them locally to run live inference against your MQTT stream.

In [17]:
os.makedirs("/kaggle/working/models", exist_ok=True)
joblib.dump(anomaly_model, "/kaggle/working/models/anomaly_detector.joblib")
joblib.dump({"model": classifier_model, "labels": labels}, "/kaggle/working/models/fault_classifier.joblib")
joblib.dump(rul_model, "/kaggle/working/models/rul_regressor.joblib")
joblib.dump(feature_cols, "/kaggle/working/models/feature_columns.joblib")
print("Saved to /kaggle/working/models/ -- download via the Output panel")

Saved to /kaggle/working/models/ -- download via the Output panel
